In [ ]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# ─────────────────────────────────────────────
# 1. LOAD THE IMPURE DATASETS
# ─────────────────────────────────────────────
df_fake = pd.read_csv("fake_impure.csv")
df_true = pd.read_csv("true_impure.csv")

# ─────────────────────────────────────────────
# 2. PRE-PROCESSING
# ─────────────────────────────────────────────

def preprocess(df, class_label):

    # --- Step 1: Fix inconsistent labels ---
    # Strip whitespace and convert to uppercase so 'true', ' TRUE', 'True  ' all become 'TRUE'
    df['label'] = df['label'].astype(str).str.strip().str.upper()

    # --- Step 2: Replace garbage / corrupt values with NaN ---
    # Any cell containing these tokens is treated as missing
    garbage_values = {'###CORRUPTED###', 'N/A', 'null', 'NULL', 'nan', 'NAN', '???', '---', ''}
    for col in ['title', 'text', 'subject']:
        df[col] = df[col].apply(
            lambda x: None if str(x).strip() in garbage_values else x
        )

    # --- Step 3: Remove duplicate rows ---
    df = df.drop_duplicates()

    # --- Step 4: Drop rows where 'text' is missing (required for model input) ---
    # For title/subject, fill with empty string so the row is still usable
    df['title']   = df['title'].fillna('')
    df['subject'] = df['subject'].fillna('')
    df = df.dropna(subset=['text'])     # text is essential — drop if missing

    # --- Step 5: Clean whitespace and newlines in text ---
    # Strip leading/trailing spaces and replace newlines with a single space
    df['text'] = df['text'].astype(str).str.strip()
    df['text'] = df['text'].str.replace(r'\n+', ' ', regex=True)
    df['text'] = df['text'].str.replace(r'\s{2,}', ' ', regex=True)  # collapse multiple spaces

    # Assign numerical class: 0 = Fake, 1 = True
    df['class'] = class_label

    return df

df_fake = preprocess(df_fake, class_label=0)
df_true = preprocess(df_true, class_label=1)

print("After preprocessing:")
print(f"  Fake rows : {len(df_fake)}")
print(f"  True rows : {len(df_true)}")

# ─────────────────────────────────────────────
# 3. EVERYTHING BELOW IS UNCHANGED FROM appp.ipynb
# ─────────────────────────────────────────────

# Combining datasets
df_mrg = pd.concat([df_fake, df_true], axis=0)

# Keep only 'text' and 'class' (drop id, title, subject, label)
df = df_mrg.drop(['id', 'title', 'subject', 'label'], axis=1)
df = df.reset_index(drop=True)

# Text cleaning function (unchanged)
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

df['text'] = df['text'].apply(clean_text)

# Defining variables
X = df['text']
y = df['class']

# Splitting data
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Vectorization
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test  = vectorization.transform(x_test)

# Training
LR = LogisticRegression()
LR.fit(xv_train, y_train)

# Evaluation
pred_lr = LR.predict(xv_test)
print(classification_report(y_test, pred_lr))

# Saving
joblib.dump(LR, 'LR_model.jb')
joblib.dump(vectorization, 'vectorizer.jb')